In [2]:
from google.cloud import aiplatform
import os
import time
import re

# === CONFIGURACIÓN ===
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../../../bubbo-dfba0-47e395cdcdc7.json"
PROJECT = "bubbo-dfba0"
LOCATION = "europe-southwest1"
STAGING_BUCKET = "gs://embeddings_new_bucket"
INDEX_NAME = "alpha_recs_movies_tv_tree_ah_eu_sw1"
EMBEDDINGS_URI = "gs://embeddings_new_bucket/embeddings/index_data/all_embeddings.jsonl"
DIMENSIONS = 768
ENDPOINT_ID = "projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272"

In [3]:
# === INICIAR ===
aiplatform.init(project=PROJECT, location=LOCATION, staging_bucket=STAGING_BUCKET)

# === ELIMINAR ÍNDICE ANTERIOR SI EXISTE ===
existing_indexes = aiplatform.MatchingEngineIndex.list(filter=f'display_name="{INDEX_NAME}"')
for idx in existing_indexes:
    print(f"Eliminando índice previo: {idx.display_name}")
    idx.delete()
    idx.wait()

# === CREAR ÍNDICE  ===
try:
    print(f"Creando nuevo índice: {INDEX_NAME}")
    index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
        display_name=INDEX_NAME,
        contents_delta_uri=EMBEDDINGS_URI,
        description="Matching Engine Index",
        dimensions=DIMENSIONS,
        approximate_neighbors_count=250,
        leaf_node_embedding_count=1000,
        distance_measure_type=aiplatform.matching_engine.matching_engine_index_config.DistanceMeasureType.COSINE_DISTANCE,
        index_update_method="STREAM_UPDATE"
    )
    index.wait()  # Esperar a que el índice se cree
    print(f"Índice {INDEX_NAME} creado correctamente.")
except Exception as e:
    print(f"Error al crear el índice: {e}")
    raise



# === DESPLEGAR AL ENDPOINT ===
try:
    endpoint = aiplatform.MatchingEngineIndexEndpoint(index_endpoint_name=ENDPOINT_ID)
    deployed_index_id = re.sub(r'[^a-zA-Z0-9_]', '_', INDEX_NAME)[:63]
    if not deployed_index_id[0].isalpha():
        deployed_index_id = f"a_{deployed_index_id}"

    # Eliminar despliegue previo con mismo ID
    for d in endpoint.deployed_indexes:
        if d.id == deployed_index_id:
            print(f"Desplegando índice previo con el mismo ID: {deployed_index_id}")
            endpoint.undeploy_index(deployed_index_id=d.id)
            time.sleep(5)

    # Desplegar el nuevo índice
    print(f"Desplegando índice al endpoint: {deployed_index_id}")
    endpoint.deploy_index(
        index=index,
        deployed_index_id=deployed_index_id,
        display_name="deployed-alpha-index"
    )

    print("✅ Índice creado, cargado y desplegado correctamente.")
except Exception as e:
    print(f"Error al desplegar el índice: {e}")
    raise


Eliminando índice previo: alpha_recs_movies_tv_tree_ah_eu_sw1
Deleting MatchingEngineIndex : projects/75629471929/locations/europe-southwest1/indexes/1888098959402991616
MatchingEngineIndex deleted. . Resource name: projects/75629471929/locations/europe-southwest1/indexes/1888098959402991616
Deleting MatchingEngineIndex resource: projects/75629471929/locations/europe-southwest1/indexes/1888098959402991616
Delete MatchingEngineIndex backing LRO: projects/75629471929/locations/europe-southwest1/indexes/1888098959402991616/operations/2627996617617178624
MatchingEngineIndex resource projects/75629471929/locations/europe-southwest1/indexes/1888098959402991616 deleted.
Creando nuevo índice: alpha_recs_movies_tv_tree_ah_eu_sw1
Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/75629471929/locations/europe-southwest1/indexes/1029037330482069504/operations/1943449474256863232
MatchingEngineIndex created. Resource name: projects/75629471929/locations/europe-southwest1/